# Lab 19 — One-click OpenAI + Neo4j Aura + Golden 50

Mục tiêu của notebook này: **fresh Colab runtime → Runtime > Run all → nhận ZIP cuối**, không cần thêm cell test/resume thủ công.

## Colab Secrets bắt buộc
- `OPENAI_API_KEY`
- `HF_TOKEN`
- `NEO4J_URI`
- `NEO4J_USERNAME` **hoặc** `NEO4J_USER` (Aura credential file thường dùng `NEO4J_USERNAME`)
- `NEO4J_PASSWORD`
- `NEO4J_DATABASE`

Optional: `LLM_MODEL` / `JUDGE_MODEL` (default `gpt-4.1-mini`). Nếu AuraDB chỉ dành riêng cho lab và an toàn để xoá graph cũ, đặt `LAB_RESET_GRAPH=1`.

Runner tự làm preflight Neo4j + OpenAI **trước khi** bắt đầu coreference/extraction tốn API; nếu preflight fail thì pipeline dừng ngay. Golden 50 chỉ chạy sau khi solution build xong Neo4j graph + FAISS indexes.

In [ ]:
#@title 1 — Clone latest main (safe to rerun)
%cd /content
!rm -rf /content/lab19
!git clone -q https://github.com/QuocKhanhLuong/K3-Track3-Lab19-GraphRAG-2A202601713-LuongQuocKhanh.git /content/lab19
%cd /content/lab19
!git log -1 --oneline


In [ ]:
#@title 2 — Load reference notebook definitions + first 5,000 source rows
import json
from pathlib import Path

reference_path = Path('/content/lab19/Day19_GraphRAG_vs_FlatRAG_Production_Lab_Guide.ipynb')
nb = json.loads(reference_path.read_text(encoding='utf-8'))

for idx, cell in enumerate(nb['cells']):
    if cell.get('cell_type') != 'code':
        continue
    src = ''.join(cell.get('source', []))

    # Official Golden 50 is based on the first 5,000 source rows.
    src = src.replace('LIMIT_ROWS = 1_000_000', 'LIMIT_ROWS = 5000')
    src = src.replace('LIMIT_MB = 300', 'LIMIT_MB = 80')
    src = src.replace('PRIORITIZE_MB = True', 'PRIORITIZE_MB = False')

    result = get_ipython().run_cell(src)
    if getattr(result, 'error_before_exec', None):
        raise result.error_before_exec
    if getattr(result, 'error_in_exec', None):
        raise result.error_in_exec

print('Reference notebook loaded. Dataset:', DATA_PATH)


In [ ]:
#@title 3 — ONE CLICK: preflight → full solution → official Golden 50 → ZIP
%run -i /content/lab19/openai_runtime_patch.py

# Automatic fail-fast before expensive LLM extraction.
preflight_services()

# Build corpus, coreference, triples, entity resolution, Neo4j graph,
# Flat FAISS index, GraphRAG index, baseline artifacts and reports.
%run -i /content/lab19/colab_solution.py

# This line is reached only if colab_solution.py completed successfully.
# The official evaluator also verifies indexes/graph state before evaluation.
%run -i /content/lab19/official_golden_eval.py


## Done
Nếu Cell 3 hoàn tất, Colab tự tải `/content/lab19_submission_official50.zip`. Hai CSV rubric chính trong `outputs/` là kết quả official Golden 50. Không chạy Golden riêng khi Cell 3 chưa hoàn tất.